<a href="https://colab.research.google.com/github/Prasanna-datascience/Data201Class/blob/main/Project1_Prasanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_auc_score
)

# ---------------------------
# Load data
# ---------------------------
heart = pd.read_csv("Heart.csv")

# Remove the unnamed index column if present
if "Unnamed: 0" in heart.columns:
    heart = heart.drop(columns="Unnamed: 0")

# Convert target to 0/1
heart["AHD"] = heart["AHD"].map({"No": 0, "Yes": 1})

# Replace text missing values
heart = heart.replace("NA", np.nan)

# ---------------------------
# Features and target
# ---------------------------
X = heart.drop(columns="AHD")
y = heart["AHD"]

# Identify column types
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

# ---------------------------
# Preprocessing
# ---------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# ---------------------------
# Train/Test Split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# ---------------------------
# Logistic Regression Model
# ---------------------------
model = Pipeline([
    ("prep", preprocessor),
    ("logistic", LogisticRegression(max_iter=500))
])

model.fit(X_train, y_train)

# ---------------------------
# Predictions
# ---------------------------
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# ---------------------------
# Evaluation
# ---------------------------
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives : {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives : {tp}")

print("\nClassification Report")
print(classification_report(y_test, y_pred))

# ---------------------------
# Lower the threshold
# ---------------------------
threshold = 0.30

y_pred_30 = (y_prob >= threshold).astype(int)

cm30 = confusion_matrix(y_test, y_pred_30)

print("\nConfusion Matrix (Threshold = 0.30)")
print(cm30)

tn, fp, fn, tp = cm30.ravel()

print(f"\nFalse Negatives at 0.30: {fn}")
print(f"False Positives at 0.30: {fp}")

print("\nClassification Report (Threshold = 0.30)")
print(classification_report(y_test, y_pred_30))